# LaneVision AI Kaggle GPU Training Workflow
This notebook prepares the existing LaneVision AI training pipeline for Kaggle GPU use without uploading the local prepared dataset.

In [ ]:
# CELL 1 - Environment and GPU verification
import json
import platform
import subprocess
import sys
from pathlib import Path

import torch

print(json.dumps({
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_count': torch.cuda.device_count(),
}, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled for this notebook session.')

In [ ]:
# CELL 2 - Repository setup
from pathlib import Path

candidates = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle')]
repo_root = None
for base in candidates:
    if base.is_file():
        continue
    search_space = [base] + [path for path in base.iterdir() if path.is_dir()] if base.exists() else []
    for path in search_space:
        if (path / 'backend' / 'training' / 'train.py').exists() and (path / 'backend' / 'requirements.txt').exists():
            repo_root = path
            break
    if repo_root is not None:
        break
if repo_root is None:
    raise FileNotFoundError('LaneVision AI repository root was not found in the current Kaggle session.')
print(repo_root)

In [ ]:
# CELL 3 - Dependency installation
import importlib
import subprocess
import sys

required_packages = {
    'fastapi': 'fastapi',
    'pydantic': 'pydantic',
    'pydantic_settings': 'pydantic-settings',
    'sqlalchemy': 'sqlalchemy',
    'multipart': 'python-multipart',
    'cv2': 'opencv-python-headless',
    'yaml': 'PyYAML',
    'httpx': 'httpx',
    'dotenv': 'python-dotenv',
}
missing = []
for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(package_name)

import torchvision
print('torch preserved:', torch.__version__)
print('torchvision:', torchvision.__version__)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
print({'installed_missing': missing})

In [ ]:
# CELL 4 - Locate TuSimple Kaggle dataset
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'backend.training.kaggle_gpu_prep', 'discover',
    '--search-root', '/kaggle/input'
], cwd=repo_root, check=True)

In [ ]:
# CELL 5 - Prepare and validate dataset
prepared_root = Path('/kaggle/working/tusimple_prepared')
subprocess.run([
    sys.executable, '-m', 'backend.training.kaggle_gpu_prep', 'prepare',
    '--search-root', '/kaggle/input',
    '--prepared-root', str(prepared_root)
], cwd=repo_root, check=True)

In [ ]:
# CELL 6 - GPU preflight and smoke test
checkpoint_dir = Path('/kaggle/working/models')
subprocess.run([
    sys.executable, '-m', 'backend.training.kaggle_gpu_prep', 'preflight',
    '--prepared-root', str(prepared_root),
    '--checkpoint-dir', str(checkpoint_dir),
    '--device', 'auto'
], cwd=repo_root, check=True)
subprocess.run([
    sys.executable, '-m', 'backend.training.kaggle_gpu_prep', 'smoke-test',
    '--prepared-root', str(prepared_root),
    '--checkpoint-dir', str(checkpoint_dir),
    '--config', 'backend/training/configs/kaggle.yaml',
    '--device', 'auto',
    '--batch-size', '2',
    '--train-batches', '20',
    '--val-batches', '10'
], cwd=repo_root, check=True)

In [ ]:
# CELL 7 - Start or resume training
train_command = [
    sys.executable, '-m', 'backend.training.train',
    '--config', 'backend/training/configs/kaggle.yaml',
    '--dataset', '/kaggle/working/tusimple_prepared',
    '--device', 'auto',
    '--batch-size', '4',
    '--epochs', '30'
]
resume_command = train_command + ['--resume', '/kaggle/working/models/lane_segmentation_last.pth']
print('Start command:', ' '.join(train_command))
print('Resume command:', ' '.join(resume_command))
# Run one of these manually after approval.

In [ ]:
# CELL 8 - Evaluate best checkpoint
eval_command = [
    sys.executable, '-m', 'backend.training.evaluate',
    '--dataset', '/kaggle/working/tusimple_prepared/test',
    '--checkpoint', '/kaggle/working/models/lane_segmentation_best.pth',
    '--device', 'auto',
    '--batch-size', '4'
]
print(' '.join(eval_command))
# subprocess.run(eval_command, cwd=repo_root, check=True)

In [ ]:
# CELL 9 - Export or download final checkpoints
export_paths = {
    'production_checkpoint': '/kaggle/working/models/lane_segmentation.pth',
    'best_checkpoint': '/kaggle/working/models/lane_segmentation_best.pth',
    'last_checkpoint': '/kaggle/working/models/lane_segmentation_last.pth',
    'training_outputs': '/kaggle/working/training_outputs',
}
print(json.dumps(export_paths, indent=2))